In [1]:
import pandas as pd
import folium
import geopandas as gpd
from shapely.geometry import Polygon, Point
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

In [ ]:
df1 = pd.read_csv(f"data/data_merged/df_large.csv",index_col=False)
df2 = pd.read_csv(f"data/data_merged/df_small.csv",index_col=False)

In [5]:
df1_unigue = df1[['LAT', 'LON']].drop_duplicates()
df2_unigue = df2[['LAT', 'LON']].drop_duplicates()

In [6]:
display(len(df1_unigue))
display(df1_unigue.head())
display(len(df2_unigue))
display(df2_unigue.head())


372

,LAT,LON
0,22.0,35.000
366,22.0,35.625
732,22.0,36.250
1098,22.0,36.875
1464,22.5,35.000


118

,LAT,LON
0,22.5,34.5
366,22.5,35.5
732,22.5,36.5
1098,23.5,34.5
1464,23.5,35.5


In [7]:
display(df1)
display(df2)

,LAT,LON,YEAR,DOY,GWETPROF,GWETROOT,GWETTOP,PRECSNO,PRECTOTCORR,QV2M,...,TS_MIN,WD2M,WD50M,WS2M,WS2M_MAX,WS2M_MIN,WS50M,WS50M_MAX,WS50M_MIN,Z0M
0,22.0,35.000,2024,1,0.31,0.31,0.24,0.0,0.00,8.42,...,9.78,28.4,32.1,2.03,3.94,1.15,4.10,5.35,3.12,0.01
1,22.0,35.000,2024,2,0.31,0.31,0.23,0.0,0.01,6.32,...,9.91,353.2,1.7,2.23,3.50,1.30,4.84,7.07,2.71,0.01
2,22.0,35.000,2024,3,0.31,0.31,0.23,0.0,0.13,7.53,...,11.55,22.4,27.5,2.68,4.56,1.60,5.32,6.80,3.05,0.01
3,22.0,35.000,2024,4,0.31,0.31,0.22,0.0,0.00,6.94,...,9.91,359.1,3.2,1.82,2.87,1.31,4.11,5.46,1.81,0.01
4,22.0,35.000,2024,5,0.31,0.31,0.21,0.0,0.05,6.69,...,9.55,13.4,24.0,1.94,2.91,1.08,4.23,6.54,1.89,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149323,32.0,34.375,2024,362,0.47,0.47,0.72,0.0,7.99,7.79,...,20.20,233.1,234.1,8.59,11.27,6.37,11.38,15.73,8.02,0.03
149324,32.0,34.375,2024,363,0.49,0.49,0.76,0.0,4.91,8.56,...,20.28,227.3,228.5,7.09,8.83,5.38,9.19,11.80,6.81,0.03
149325,32.0,34.375,2024,364,0.49,0.49,0.75,0.0,9.91,8.71,...,20.28,231.9,233.0,6.30,6.69,5.94,7.97,8.47,7.23,0.03
149326,32.0,34.375,2024,365,0.50,0.51,0.79,0.0,11.64,8.84,...,20.14,257.1,257.8,6.16,7.17,5.40,7.77,9.36,6.64,0.03


,LAT,LON,YEAR,DOY,AIRMASS,ALLSKY_SFC_SW_DWN,ALLSKY_SFC_UVA,ALLSKY_SFC_UVB,ALLSKY_SFC_UV_INDEX,CLOUD_AMT,CLOUD_AMT_DAY,CLOUD_AMT_NIGHT,CLRSKY_DAYS,MIDDAY_INSOL,PSH,PW
0,22.5,34.5,2024,1,4.11,18.01,0.98,0.02,1.45,1.72,2.53,1.03,1.0,66.30,0.43,0.90
1,22.5,34.5,2024,2,4.10,18.19,0.98,0.02,1.44,0.17,0.27,0.09,1.0,66.99,0.44,0.78
2,22.5,34.5,2024,3,4.10,18.04,0.98,0.03,1.53,0.50,0.64,0.38,1.0,66.82,0.43,0.88
3,22.5,34.5,2024,4,4.10,17.81,0.98,0.03,1.53,0.83,0.44,1.16,1.0,65.76,0.43,1.20
4,22.5,34.5,2024,5,4.10,18.00,0.99,0.03,1.54,2.22,3.44,1.19,1.0,66.62,0.43,1.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44647,31.5,34.5,2024,362,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.0,-999.00,-999.00,-999.00
44648,31.5,34.5,2024,363,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.0,-999.00,-999.00,-999.00
44649,31.5,34.5,2024,364,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.0,-999.00,-999.00,-999.00
44650,31.5,34.5,2024,365,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.00,-999.0,-999.00,-999.00,-999.00


In [8]:
egypt_boundary = gpd.read_file("https://raw.githubusercontent.com/datasets/geo-boundaries-world-110m/master/countries.geojson")

egypt_polygon = egypt_boundary[egypt_boundary['admin'] == 'Egypt'].geometry.iloc[0]
coords = list(egypt_polygon.exterior.coords)
updated_coords = coords 
updated_egypt_polygon = Polygon(updated_coords)


In [9]:
egypt_map = folium.Map(location=[26.5, 30], zoom_start=6)
points_within_polygon = []
for _, row in df1_unigue.iterrows():
    lat, lon = row['LAT'], row['LON']
    
    
    point = gpd.points_from_xy([lon], [lat])[0]
    
    if updated_egypt_polygon.contains(point): 
        points_within_polygon.append((lat, lon)) 
        # Add rectangle
        folium.CircleMarker(
    location=[lat, lon],
    radius=4,  
    color='blue',
    fill=True,
    fill_color='blue',
    fill_opacity=0.8,
    tooltip=f"Lat: {lat}, Lon: {lon}",
    popup=f"Coordinates: Lat {lat}, Lon {lon}"
).add_to(egypt_map)


specific_points = [(33.75, 27), (33.75, 27.5),(33.75,31.0),(34.375,26.0),(26.875,31.5),(28.75,31.0),(35.0,25.0),(35.625,23.5),(34.375,28.0),(34.375,28.5),(25.0,31.5),(32.5,29.5),(31.875,31.5)]

for lon, lat in specific_points:
    folium.CircleMarker(
    location=[lat, lon],
    radius=4,
    color='red',
    fill=True,
    fill_color='blue',
    fill_opacity=0.8,
    tooltip=f"Lat: {lat}, Lon: {lon}",
    popup=f"Coordinates: Lat {lat}, Lon {lon}"
).add_to(egypt_map)

egypt_map


In [10]:
total_points_in_Egypt=points_within_polygon+specific_points

df3 = pd.DataFrame(total_points_in_Egypt, columns=['Latitude', 'Longitude'])

display(df3)

,Latitude,Longitude
0,22.500,35.000
1,22.500,35.625
2,22.500,36.250
3,23.000,35.000
4,23.000,35.625
...,...,...
284,34.375,28.000
285,34.375,28.500
286,25.000,31.500
287,32.500,29.500


In [11]:
egypt_map_small = folium.Map(location=[26.5, 30], zoom_start=6)
points_within_polygon_small = []
for _, row in df2_unigue.iterrows():
    lat, lon = row['LAT'], row['LON']
    
    point_small = gpd.points_from_xy([lon], [lat])[0]
    
    if updated_egypt_polygon.contains(point_small): 
        points_within_polygon_small.append((lat, lon)) 

        folium.CircleMarker(
    location=[lat, lon],
    radius=4, 
    color='blue',
    fill=True,
    fill_color='blue',
    fill_opacity=0.8,
    tooltip=f"Lat: {lat}, Lon: {lon}",
    popup=f"Coordinates: Lat {lat}, Lon {lon}"
).add_to(egypt_map_small)

specific_points_small = [(27.5,31.5),(32.5,29.5),(31.5,31.5),(32.5,31.5),(33.5,27.5),(35.5,24.5)]

for lon, lat in specific_points_small:
    folium.CircleMarker(
    location=[lat, lon],
    radius=4, 
    color='red',
    fill=True,
    fill_color='blue',
    fill_opacity=0.8,
    tooltip=f"Lat: {lat}, Lon: {lon}",
    popup=f"Coordinates: Lat {lat}, Lon {lon}"
).add_to(egypt_map_small)

#egypt_map_small.save("egypt_grid_map_small.html")

egypt_map_small


In [12]:
total_points_in_Egypt_small=points_within_polygon_small+specific_points_small

df5 = pd.DataFrame(total_points_in_Egypt_small, columns=['Latitude', 'Longitude'])

print(df5)

    Latitude  Longitude
0       22.5       34.5
1       22.5       35.5
2       23.5       34.5
3       23.5       35.5
4       24.5       34.5
..       ...        ...
91      32.5       29.5
92      31.5       31.5
93      32.5       31.5
94      33.5       27.5
95      35.5       24.5

[96 rows x 2 columns]


In [13]:
df_filtered_small = df2[df2.apply(lambda row: (row['LAT'], row['LON']) in zip(df5['Latitude'], df5['Longitude']), axis=1)]

print("Filtered DataFrame (rows matching Latitude and Longitude pairs in df3):")
print(df_filtered_small)

Filtered DataFrame (rows matching Latitude and Longitude pairs in df3):
        LAT   LON  YEAR  DOY  AIRMASS  ALLSKY_SFC_SW_DWN  ALLSKY_SFC_UVA  \
0      22.5  34.5  2024    1     4.11              18.01            0.98   
1      22.5  34.5  2024    2     4.10              18.19            0.98   
2      22.5  34.5  2024    3     4.10              18.04            0.98   
3      22.5  34.5  2024    4     4.10              17.81            0.98   
4      22.5  34.5  2024    5     4.10              18.00            0.99   
...     ...   ...   ...  ...      ...                ...             ...   
43549  31.5  31.5  2024  362  -999.00            -999.00         -999.00   
43550  31.5  31.5  2024  363  -999.00            -999.00         -999.00   
43551  31.5  31.5  2024  364  -999.00            -999.00         -999.00   
43552  31.5  31.5  2024  365  -999.00            -999.00         -999.00   
43553  31.5  31.5  2024  366  -999.00            -999.00         -999.00   

       ALLSKY_S

In [14]:
df_sorted_small = df_filtered_small.sort_values(by=['YEAR', 'DOY',"LAT","LON"], ascending=[True, True,True,True])

print(df_sorted_small)

df_sorted_small.to_csv('data/sorted_data_small.csv', index=False)


        LAT   LON  YEAR  DOY  AIRMASS  ALLSKY_SFC_SW_DWN  ALLSKY_SFC_UVA  \
4758   22.5  25.5  2024    1     3.93              18.37            1.00   
5124   22.5  26.5  2024    1     4.02              17.90            0.98   
5490   22.5  27.5  2024    1     4.15              17.28            0.95   
5856   22.5  28.5  2024    1     4.30              17.12            0.94   
6222   22.5  29.5  2024    1     4.47              17.16            0.94   
...     ...   ...   ...  ...      ...                ...             ...   
40625  30.5  34.5  2024  366  -999.00            -999.00         -999.00   
41357  31.5  25.5  2024  366  -999.00            -999.00         -999.00   
41723  31.5  26.5  2024  366  -999.00            -999.00         -999.00   
43187  31.5  30.5  2024  366  -999.00            -999.00         -999.00   
43553  31.5  31.5  2024  366  -999.00            -999.00         -999.00   

       ALLSKY_SFC_UVB  ALLSKY_SFC_UV_INDEX  CLOUD_AMT  CLOUD_AMT_DAY  \
4758           

In [15]:

df_filtered_large = df1[df1.apply(lambda row: (row['LAT'], row['LON']) in zip(df3['Latitude'], df3['Longitude']), axis=1)]
print("Filtered DataFrame (rows matching Latitude and Longitude pairs in df3):")
print(df_filtered_large)

Filtered DataFrame (rows matching Latitude and Longitude pairs in df3):
         LAT    LON  YEAR  DOY  GWETPROF  GWETROOT  GWETTOP  PRECSNO  \
1464    22.5  35.00  2024    1      0.34      0.34     0.28      0.0   
1465    22.5  35.00  2024    2      0.34      0.34     0.27      0.0   
1466    22.5  35.00  2024    3      0.34      0.34     0.27      0.0   
1467    22.5  35.00  2024    4      0.34      0.34     0.26      0.0   
1468    22.5  35.00  2024    5      0.34      0.34     0.25      0.0   
...      ...    ...   ...  ...       ...       ...      ...      ...   
141637  31.5  31.25  2024  362      0.60      0.59     0.30      0.0   
141638  31.5  31.25  2024  363      0.60      0.59     0.31      0.0   
141639  31.5  31.25  2024  364      0.60      0.59     0.34      0.0   
141640  31.5  31.25  2024  365      0.60      0.59     0.35      0.0   
141641  31.5  31.25  2024  366      0.60      0.59     0.35      0.0   

        PRECTOTCORR  QV2M  ...  TS_MIN   WD2M  WD50M  WS2M  WS2

In [22]:
shapefile_path = "City_ADM\EGY_adm2.shp"
cities = gpd.read_file(shapefile_path).to_crs("EPSG:3857") 
def find_nearest_city(point, cities):
    """Find the nearest city to a given point (in projected CRS)."""
    distances = cities.distance(point)
    nearest_idx = distances.idxmin()
    return cities.loc[nearest_idx, city_column] if nearest_idx in cities.index else None

def find_nearest_governorate(point, cities_gdf):
    """Find the governorate of the nearest city for a given point."""
    distances = cities_gdf.geometry.distance(point)
    nearest_idx = distances.idxmin()
    if pd.notna(nearest_idx):
        return cities_gdf.loc[nearest_idx, "NAME_1"] 
    return None 


FILES=["large.csv","small.csv"]
for i in FILES:
    df = pd.read_csv("data/sorted_data_{}".format(i))

    df["geometry"] = df.apply(lambda row: Point(row["LON"], row["LAT"]), axis=1)
    geo_df = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

    geo_df = geo_df.to_crs("EPSG:3857")

    result = gpd.sjoin(geo_df, cities, how="left", predicate="within")

    city_column = "NAME_2" 
    result = result.rename(columns={city_column: "City"})
    GOV_column = "NAME_1" 
    result = result.rename(columns={GOV_column: "GOV"})
    
    
    missing_city_mask = result["City"].isna()
    if missing_city_mask.any():
        print(f"Fixing {missing_city_mask.sum()} missing locations...")
        result.loc[missing_city_mask, "City"] = result.loc[missing_city_mask, "geometry"].apply(lambda x: find_nearest_city(x, cities))
        
    missing_gov_mask = result["GOV"].isna()
    if missing_gov_mask.any():
        print(f"Fixing {missing_gov_mask.sum()} missing governorates...")
        result.loc[missing_gov_mask, "GOV"] = result.loc[missing_gov_mask, "geometry"].apply(
            lambda x: find_nearest_governorate(x, cities)
        )

    result["Date"] = pd.to_datetime(result["YEAR"].astype(str) + result["DOY"].astype(str), format='%Y%j')
    result["Week"] = result["Date"].dt.isocalendar().week  
    result["Month"] = result["Date"].dt.month
    print("Missing governorates after fixing:", result["GOV"].isna().sum())
    result.to_csv("data/fullWithLocations_{}".format(i), index=False)    

Fixing 4758 missing locations...
Fixing 4758 missing governorates...
Missing governorates after fixing: 0
Fixing 1830 missing locations...
Fixing 1830 missing governorates...
Missing governorates after fixing: 0


In [ ]:
FILES=["large.csv","small.csv"]
for i in FILES:
    df = pd.read_csv("data/fullWithLocations_{}".format(i))
    display(df.columns)
    df = df.drop(columns=["geometry", "index_right", "ID_0", "ISO", "NAME_0", "ID_1", "ID_2", "TYPE_2", "ENGTYPE_2", "NL_NAME_2", "VARNAME_2"])
    df.to_csv("data/fullWithLocations_FINAL_{}".format(i), index=False)
os.remove("data/data_merged/df_small.csv")
os.remove("data/data_merged/df_large.csv")
os.rmdir("data/data_merged")  

Index(['LAT', 'LON', 'YEAR', 'DOY', 'GWETPROF', 'GWETROOT', 'GWETTOP',
       'PRECSNO', 'PRECTOTCORR', 'QV2M', 'RH2M', 'RHOA', 'T10M', 'T10M_MAX',
       'T10M_MIN', 'T2M', 'T2M_MAX', 'T2M_MIN', 'TO3', 'TS', 'TSOIL1',
       'TSOIL2', 'TSOIL3', 'TSOIL4', 'TSOIL5', 'TSOIL6', 'TS_MAX', 'TS_MIN',
       'WD2M', 'WD50M', 'WS2M', 'WS2M_MAX', 'WS2M_MIN', 'WS50M', 'WS50M_MAX',
       'WS50M_MIN', 'Z0M', 'geometry', 'index_right', 'ID_0', 'ISO', 'NAME_0',
       'ID_1', 'GOV', 'ID_2', 'City', 'TYPE_2', 'ENGTYPE_2', 'NL_NAME_2',
       'VARNAME_2', 'Date', 'Week', 'Month'],
      dtype='object')

Index(['LAT', 'LON', 'YEAR', 'DOY', 'AIRMASS', 'ALLSKY_SFC_SW_DWN',
       'ALLSKY_SFC_UVA', 'ALLSKY_SFC_UVB', 'ALLSKY_SFC_UV_INDEX', 'CLOUD_AMT',
       'CLOUD_AMT_DAY', 'CLOUD_AMT_NIGHT', 'CLRSKY_DAYS', 'MIDDAY_INSOL',
       'PSH', 'PW', 'geometry', 'index_right', 'ID_0', 'ISO', 'NAME_0', 'ID_1',
       'GOV', 'ID_2', 'City', 'TYPE_2', 'ENGTYPE_2', 'NL_NAME_2', 'VARNAME_2',
       'Date', 'Week', 'Month'],
      dtype='object')

In [ ]:
data_small_df = pd.read_csv("data/fullWithLocations_FINAL_small.csv")

data_large_df = pd.read_csv("data/fullWithLocations_FINAL_large.csv")

In [ ]:
#######################################################################
group_small = [
    "AIRMASS", "ALLSKY_SFC_SW_DWN", "ALLSKY_SFC_UVA", "ALLSKY_SFC_UVB",
    "ALLSKY_SFC_UV_INDEX", "CLOUD_AMT", "CLOUD_AMT_DAY", "CLOUD_AMT_NIGHT",
    "CLRSKY_DAYS", "MIDDAY_INSOL", "PSH", "PW"
]

group_large = [
    " EVLAND", "GWETPROF", "GWETROOT", "GWETTOP", "PRECSNO", "PRECTOTCORR",
    "QV2M", "RH2M", "RHOA", "T10M", "T10M_MAX", "T10M_MIN", "T2M",
    "T2M_MAX", "T2M_MIN", "TO3", "TS", "TSOIL1", "TSOIL2", "TSOIL3",
    "TSOIL4", "TSOIL5", "TSOIL6", "TS_MAX", "TS_MIN", "WD2M", "WD50M",
    "WS2M", "WS2M_MAX", "WS2M_MIN", "WS50M", "WS50M_MAX", "WS50M_MIN", "Z0M"
]


In [ ]:
loc_Small=data_small_df[["LON","LAT"]].drop_duplicates()
loc_Large=data_large_df[["LON","LAT"]].drop_duplicates()

display(loc_Large[["LAT","LON"]].drop_duplicates().head(10))
display(loc_Small[["LAT","LON"]].drop_duplicates().head(10))
display(len(loc_Large))

In [ ]:
loc_Large = loc_Large[loc_Large["LAT"].isin(loc_Small["LAT"])]
mask = loc_Large["LON"].apply(lambda x: np.any(np.abs(loc_Small["LON"] - x) <= 0.25))
loc_Large = loc_Large[mask]

In [ ]:
loc_Large['LON'] = loc_Large['LON'].astype(float)

lon_mapping = {
    25.625: 25.5,
    26.25: 26.5,
    27.5: 27.5,
    28.75: 28.5,
    29.375: 29.5,
    30.625: 30.5,
    31.25: 31.5,
    32.5: 32.5,
    33.75: 33.5,
    34.375: 34.5,
    35.625: 35.5
}
loc_Large['LON'].replace(lon_mapping, inplace=True)
loc_Large.dropna()

In [ ]:
loc_Small=pd.read_csv("data/LocSmall.csv")
loc_Large=pd.read_csv("data/LocLarge.csv")
egypt_map = folium.Map(location=[26.5, 30], zoom_start=6)


for _, row in loc_Large[["LAT", "LON"]].drop_duplicates().iterrows():
    folium.Rectangle(
        bounds=[
            [row['LAT'] - 0.25, row['LON'] - 0.3125],
            [row['LAT'] + 0.25, row['LON'] + 0.3125]
        ],
        color='black',
        fill=True,
        fill_opacity=0.3,
        weight=1,
        tooltip=f"Grid Cell<br>Center LAT: {row['LAT']:.3f}<br>Center LON: {row['LON']:.3f}"
    ).add_to(egypt_map)
for _, row in loc_Large[["LAT", "LON"]].drop_duplicates().iterrows():
    folium.Circle(
        location=[row['LAT'], row['LON']],
        radius=200,
        color='blue',
        fill=True,
        fill_opacity=0.1,
        tooltip=f"Large Dataset Point<br>LAT: {row['LAT']:.3f}<br>LON: {row['LON']:.3f}",
        fill_color='blue'
    ).add_to(egypt_map)

for _, row in loc_Small[["LAT", "LON"]].drop_duplicates().iterrows():
    folium.Circle(
        location=[row['LAT'], row['LON']],
        radius=250,
        color='red',
        fill=True,
        tooltip=f"Small Dataset Point<br>LAT: {row['LAT']:.3f}<br>LON: {row['LON']:.3f}",
        fill_color='red',
        fill_opacity=0.1
    ).add_to(egypt_map)
egypt_map



In [ ]:
print("df_small ",loc_Small.columns)
print("df_large ",loc_Large.columns)

In [ ]:
df_small = loc_Small.drop(columns=['Unnamed: 0'], errors='ignore')
df_small = df_small.dropna()

df_large = loc_Large.drop(columns=['Unnamed: 0'], errors='ignore')

merged_df = pd.merge(df_small, df_large, on=['LAT', 'LON', 'DOY', 'YEAR'], how='inner')

merged_df = merged_df.drop_duplicates(subset=['LAT', 'LON', 'DOY', 'YEAR'])


In [ ]:
print(len(merged_df  ))

In [ ]:
merged_df[['LAT', 'LON','GOV_x', 'City_x','DOY',
 'Date_x','Week_x','Month_x',
 'YEAR',
 'AIRMASS',
 'ALLSKY_SFC_SW_DWN',
 'ALLSKY_SFC_UVA',
 'ALLSKY_SFC_UVB',
 'ALLSKY_SFC_UV_INDEX',
 'CLOUD_AMT',
 'CLOUD_AMT_DAY',
 'CLOUD_AMT_NIGHT',
 'CLRSKY_DAYS','EVLAND','GWETPROF',
 'GWETROOT',
 'GWETTOP','MIDDAY_INSOL','PRECSNO',
 'PRECTOTCORR',
 'PSH',
 'PW',
 'QV2M',
 'RH2M',
 'RHOA',
 'T10M',
 'T10M_MAX',
 'T10M_MIN',
 'T2M',
 'T2M_MAX',
 'T2M_MIN',
 'TO3',
 'TS',
 'TSOIL1',
 'TSOIL2',
 'TSOIL3',
 'TSOIL4',
 'TSOIL5',
 'TSOIL6',
 'TS_MAX',
 'TS_MIN',
 'WD2M',
 'WD50M',
 'WS2M',
 'WS2M_MAX',
 'WS2M_MIN',
 'WS50M',
 'WS50M_MAX',
 'WS50M_MIN','Z0M']].to_csv("data/Applications.csv",index=False)

In [ ]:
merged_df.to_csv("data/Applications.csv",index=False)